# Georgia ERA5 -> PRISM Temperature Downscaling Demo

This notebook is the professor-facing demonstration of the baseline pipeline:

- ERA5 coarse temperature input
- PRISM higher-resolution daily target
- CNN baseline downscaler

It is designed to match the project scripts exactly.


## 1) Imports and Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import Markdown, display, Image

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from datasets.prism_dataset import ERA5_PRISM_Dataset
from models.cnn_downscaler import CNNDownscaler


## 2) Paths

In [ ]:
ERA5_PATH = ROOT / 'data_raw' / 'era5_georgia_temp.nc'
PRISM_PATH = ROOT / 'data_raw' / 'prism'
CHECKPOINT_PATH = ROOT / 'checkpoints' / 'cnn_downscaler_best.pt'
RESULTS_DIR = ROOT / 'results' / 'evaluation'

print('ERA5_PATH:', ERA5_PATH)
print('PRISM_PATH:', PRISM_PATH)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)
print('RESULTS_DIR:', RESULTS_DIR)


## 3) Load Dataset

The dataset handles ERA5 daily aggregation, PRISM raster loading, CRS handling, clipping, and date alignment via `YYYYMMDD` in filenames.


In [ ]:
if not ERA5_PATH.exists():
    raise FileNotFoundError(f'ERA5 file not found: {ERA5_PATH}. Run data_pipeline/download_era5_georgia.py first.')
if not PRISM_PATH.exists():
    raise FileNotFoundError(f'PRISM directory not found: {PRISM_PATH}. Run data_pipeline/download_prism.py first.')

dataset = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_PATH))
x0, y0 = dataset[0]
meta0 = dataset.metadata(0)

print('Aligned samples:', len(dataset))
print('Sample date:', meta0.date.date())
print('ERA5 tensor shape:', tuple(x0.shape))
print('PRISM tensor shape:', tuple(y0.shape))
print('Example PRISM file:', meta0.prism_path)


## 4) Load Trained CNN

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        f'Checkpoint not found: {CHECKPOINT_PATH}. '
        'Run training/train_downscaler.py first.'
    )

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    model_cfg = checkpoint.get('model_config', {})
    model = CNNDownscaler(
        in_channels=model_cfg.get('in_channels', 1),
        out_channels=model_cfg.get('out_channels', 1),
        base_channels=model_cfg.get('base_channels', 32),
    )
    model.load_state_dict(checkpoint['model_state_dict'])
else:
    model = CNNDownscaler()
    model.load_state_dict(checkpoint)

model = model.to(device)
model.eval()
print('Model loaded on', device)


## 5) Inference + Metrics (few samples)

In [ ]:
num_eval = min(4, len(dataset))
rmse_scores = []
mae_scores = []
samples = []

with torch.no_grad():
    for idx in range(num_eval):
        x, y = dataset[idx]
        date = dataset.metadata(idx).date
        x_batch = x.unsqueeze(0).to(device)
        y_batch = y.unsqueeze(0).to(device)
        pred = model(x_batch, target_size=(y.shape[-2], y.shape[-1]))
        rmse = torch.sqrt(torch.mean((pred - y_batch) ** 2)).item()
        mae = torch.mean(torch.abs(pred - y_batch)).item()
        rmse_scores.append(rmse)
        mae_scores.append(mae)
        samples.append((date, x.cpu(), pred.squeeze(0).cpu(), y.cpu()))

print(f'RMSE (mean over {num_eval} samples): {np.mean(rmse_scores):.4f}')
print(f'MAE  (mean over {num_eval} samples): {np.mean(mae_scores):.4f}')


## 6) Visual Comparison (ERA5 vs Prediction vs PRISM)

In [ ]:
date, era5_tensor, pred_tensor, target_tensor = samples[0]

era5_up = F.interpolate(
    era5_tensor.unsqueeze(0),
    size=(target_tensor.shape[-2], target_tensor.shape[-1]),
    mode='bilinear',
    align_corners=False,
).squeeze(0)

era5_map = era5_up.squeeze().numpy()
pred_map = pred_tensor.squeeze().numpy()
target_map = target_tensor.squeeze().numpy()

vmin = min(era5_map.min(), pred_map.min(), target_map.min())
vmax = max(era5_map.max(), pred_map.max(), target_map.max())

fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
im = axes[0].imshow(era5_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[0].set_title('ERA5 Input (Upsampled)')
axes[0].axis('off')

axes[1].imshow(pred_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[1].set_title('CNN Prediction')
axes[1].axis('off')

axes[2].imshow(target_map, cmap='coolwarm', vmin=vmin, vmax=vmax)
axes[2].set_title('PRISM Target')
axes[2].axis('off')

fig.suptitle(f'Georgia Downscaling Sample | {date.date()}')
fig.colorbar(im, ax=axes, shrink=0.8, label='Temperature (deg C)')
plt.show()


## 7) Saved Evaluation Output (if available)

In [ ]:
plot_files = sorted(RESULTS_DIR.glob('comparison_*.png'))

if plot_files:
    display(Markdown('### Example figure generated by evaluation/evaluate_model.py'))
    display(Image(filename=str(plot_files[0])))
else:
    cmd = ('python evaluation/evaluate_model.py --era5-path data_raw/era5_georgia_temp.nc '           '--prism-path data_raw/prism --checkpoint-path checkpoints/cnn_downscaler_best.pt '           '--num-samples 8 --num-plots 1 --results-dir results/evaluation')
    display(Markdown(f'No saved evaluation plot found. Run:\n\n`{cmd}`'))


## 8) Conclusion and Next Step

This baseline establishes a clean ERA5->PRISM workflow for Georgia.

Next research step: move beyond this CNN baseline to stronger spatiotemporal and transformer-style climate models inspired by Prithvi WxC ideas.
